# Chapter 9 Exercise 2 Solution - Low-Energy Cavity

This notebook answers the exercise in two ways:

1. direct calculation in the short-cavity limit;
2. actual SciBmad simulation with `RFCavity` and `track!`.

Both parts use exactly the same numerical parameters. No separate reference-section or
`Marker`-based calculation is used.

In [5]:
using SciBmad
using Printf

## Unified chosen parameters

The exercise does not supply numerical values, so this worked solution chooses:

$$
z_{\mathrm{in}}=0.01\ \mathrm{m},\qquad
p_{c,\mathrm{in}}=0.5mc^2,\qquad
\Delta E=V=0.5mc^2.
$$

The particle is a proton. The initial momentum is low enough that its initial velocity is
significantly below $c$. The SciBmad cavity uses a very short but nonzero length
$L=10^{-8}$ m to approximate the instantaneous cavity assumed by the direct calculation.
Its RF frequency is 1 MHz, and `phi0` is calculated so that the chosen particle reaches the
cavity at the accelerating crest. Because the kick is deliberately large, 100 integration
steps are used.

In [6]:
species = Species("proton")
mass_energy = massof(species)

# Values chosen for this worked solution and shared by both calculations.
z_in = 0.01
pc_in = 0.5 * mass_energy
delta_E = 0.5 * mass_energy

# Parameters needed only to represent the short cavity in SciBmad.
cavity_L = 1e-8
rf_frequency = 1e6

E_in = sqrt(pc_in^2 + mass_energy^2)
beta_in = pc_in / E_in
omega_rf = 2pi * rf_frequency

# SciBmad uses phi_particle = phi0 - omega*z/(beta*c).
# This choice puts the selected particle at phi_particle = pi/2, giving ΔE = V.
phi0 = pi/2 + omega_rf * z_in / (beta_in * C_LIGHT)

1.5712649719888354

## 1. Direct short-cavity calculation

For an instantaneous cavity, the arrival-time difference remains unchanged. Since
$z=-\beta c\Delta t$,

$$
z_{\mathrm{out}}=z_{\mathrm{in}}
\frac{\beta_{\mathrm{out}}}{\beta_{\mathrm{in}}}.
$$

SciBmad's `RFCavity` keeps the beamline reference momentum fixed. Therefore, the direct
prediction for its final longitudinal momentum coordinate is

$$
p_{z,\mathrm{out}}=\frac{p_{c,\mathrm{out}}-p_{c,\mathrm{in}}}{p_{c,\mathrm{in}}}.
$$

In [7]:
E_out_direct = E_in + delta_E
pc_out_direct = sqrt(E_out_direct^2 - mass_energy^2)
beta_out_direct = pc_out_direct / E_out_direct

z_out_direct = z_in * beta_out_direct / beta_in
pz_out_direct = (pc_out_direct - pc_in) / pc_in

@printf("Proton rest energy = %.9e eV\n", mass_energy)
@printf("Initial pc         = %.9e eV\n", pc_in)
@printf("Energy gain        = %.9e eV = 0.5 mc^2\n", delta_E)
@printf("Initial beta       = %.12f\n", beta_in)
@printf("Final beta         = %.12f\n", beta_out_direct)
@printf("Direct final z     = %.12f m\n", z_out_direct)
@printf("Direct final pz    = %.12f\n", pz_out_direct)

Proton rest energy = 9.382720894e+08 eV
Initial pc         = 4.691360447e+08 eV
Energy gain        = 4.691360447e+08 eV = 0.5 mc^2
Initial beta       = 0.447213595500
Final beta         = 0.786151377757
Direct final z     = 0.017578879213 m
Direct final pz    = 1.544039299028


## 2. Actual SciBmad `RFCavity` simulation

The following code constructs the cavity, places it in a beamline, creates a particle, and
actually tracks the particle through the cavity with `track!`. The cavity voltage is the same
$0.5mc^2$ energy gain used in the direct calculation.

In [8]:
cavity = RFCavity(
    name="LOW_ENERGY_RF_CAVITY",
    L=cavity_L,
    voltage=delta_E,
    rf_frequency=rf_frequency,
    phi0=phi0,
    tracking_method=SciBmad.SplitIntegration(order=2, num_steps=100),
)

cavity_line = Beamline([cavity]; species_ref=species, pc_ref=pc_in)
initial_coords = [0.0, 0.0, 0.0, 0.0, z_in, 0.0]
particle = Bunch(
    copy(initial_coords);
    species=cavity_line.species_ref,
    p_over_q_ref=cavity_line.p_over_q_ref,
)

track!(particle, cavity_line)
final_coords = copy(particle.coords.v[1, :])
z_out_scibmad = final_coords[5]
pz_out_scibmad = final_coords[6]

@printf("SciBmad final z    = %.12f m\n", z_out_scibmad)
@printf("SciBmad final pz   = %.12f\n", pz_out_scibmad)
@printf("|Delta z|          = %.3e m\n", abs(z_out_scibmad - z_out_direct))
@printf("|Delta pz|         = %.3e\n", abs(pz_out_scibmad - pz_out_direct))

@assert isapprox(z_out_scibmad, z_out_direct; atol=1e-8)
@assert isapprox(pz_out_scibmad, pz_out_direct; atol=1e-10)

SciBmad final z    = 0.017578884653 m
SciBmad final pz   = 1.544039299028
|Delta z|          = 5.440e-09 m
|Delta pz|         = 3.353e-14


## Result

The direct calculation and actual SciBmad tracking now use one consistent set of values:

$$
p_{c,\mathrm{in}}=0.5mc^2,\qquad \Delta E=V=0.5mc^2.
$$

Their final $p_z$ values agree to numerical precision. Their final $z$ values differ only by
a few nanometers because the direct calculation assumes a zero-length cavity, whereas SciBmad
requires a small nonzero cavity length for this tracking method.